In [1]:
from pydash import filter_

from lib.sources import load_faq_data


predicate = {"course": "llm-zoomcamp"}

documents = filter_(load_faq_data(), predicate)

In [2]:
print(documents[0]["id"])
print(documents[0]["course"])
print(documents[0]["section"])
print(documents[0]["question"])
print(documents[0]["answer"])

74eb249bbf
llm-zoomcamp
General Course-Related Questions
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [ ]:
from pydantic import BaseModel


class Questions(BaseModel):
    questions: list[str]

In [4]:
how_to_generate_questions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [5]:
from dotenv import dotenv_values
from openai import OpenAI


config = dotenv_values()

openai_client = OpenAI(api_key=config["OPENAI_API_KEY"])

In [6]:
import json


user_prompt = json.dumps(documents[0])

In [7]:
from openai.types.responses import ResponseInputParam


messages: ResponseInputParam = [
    {"role": "developer", "content": how_to_generate_questions},
    {"role": "user", "content": user_prompt}
]

In [8]:
response = openai_client.responses.parse(
    model="gpt-4.1-mini",
    input=messages,
    text_format=Questions,
)

In [9]:
direct_result = response.output_parsed

assert direct_result is not None

print(direct_result)

questions=['Is it possible to join the course after it has already started?', 'Can I still get a certificate if I join the course late?', 'What do I need to do to earn a certificate if I join now?', 'Are project submissions still open for new students joining late?', 'If I just found out about the course, how can I catch up and get a certificate?']


In [10]:
from lib.llm import call_structured_llm


result, usage = call_structured_llm(
    openai_client,
    instructions=how_to_generate_questions,
    user_prompt=user_prompt,
    output_type=Questions
)

print("Questions:")
for question in result.questions:
    print(question)


print(f"Cost - Input: {usage.input_tokens}, Output: {usage.output_tokens}")

Questions:
I just found this course — can I still sign up and take part now?
Am I allowed to join the course late if I only discovered it recently?
If I start the course after it already began, can I still get a certificate?
What do I need to do to earn the certificate if I join the course now?
Is it too late to enroll, or can I still submit the project while submissions are open?
Cost - Input: 207, Output: 100


In [11]:
from lib.llm import calc_price


cost = calc_price(usage)

print(cost)

UsagePrice(input_cost=0.00015525, output_cost=0.00045000000000000004, total_cost=0.0006052500000000001)


In [12]:
documents[0]

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [13]:
doc = documents[0]

records: list[dict[str, str]] = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

print(records)

[{'question': 'I just found this course — can I still sign up and take part now?', 'document': '74eb249bbf'}, {'question': 'Am I allowed to join the course late if I only discovered it recently?', 'document': '74eb249bbf'}, {'question': 'If I start the course after it already began, can I still get a certificate?', 'document': '74eb249bbf'}, {'question': 'What do I need to do to earn the certificate if I join the course now?', 'document': '74eb249bbf'}, {'question': 'Is it too late to enroll, or can I still submit the project while submissions are open?', 'document': '74eb249bbf'}]


In [ ]:
from typing import TypedDict

from openai.types.responses import ResponseUsage

from lib.llm import call_structured_llm_with_retry
from lib.sources import FAQDocument


class GroundTruthRecord(TypedDict):
    question: str
    document: str


def generate_ground_truth(
    doc: FAQDocument,
) -> tuple[list[GroundTruthRecord], ResponseUsage]:

    user_prompt = json.dumps(doc)

    out, usage = call_structured_llm_with_retry(
        openai_client,
        instructions=how_to_generate_questions,
        user_prompt=user_prompt,
        output_type=Questions
    )

    results: list[GroundTruthRecord] = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [15]:
from tqdm.auto import tqdm


ground_truth: list[GroundTruthRecord] = []
usages: list[ResponseUsage] = []

for doc in tqdm(documents[:5]):
    generated_records, usage = generate_ground_truth(doc)
    ground_truth.extend(generated_records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

In [16]:
for _ in ground_truth:
    print(_)

{'question': 'I just found this course — am I still able to join now, or is it too late?', 'document': '74eb249bbf'}
{'question': 'Can new students still sign up after the course has already started?', 'document': '74eb249bbf'}
{'question': 'If I start the course late, can I still get a certificate somehow?', 'document': '74eb249bbf'}
{'question': 'What’s the deadline if I want to be eligible for the certificate after joining late?', 'document': '74eb249bbf'}
{'question': 'Is it okay to join the course now, and what do I need to do to earn the certificate?', 'document': '74eb249bbf'}
{'question': 'I signed up for the LLM Zoomcamp, but I never got any confirmation email — is that a problem, or am I already in?', 'document': '977bf7786c'}
{'question': 'Do I actually need to wait for an acceptance or confirmation message before I can start the LLM Zoomcamp materials and homework?', 'document': '977bf7786c'}
{'question': 'If I registered for the course, does that mean my name has to be on 

In [17]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress


with ThreadPoolExecutor(max_workers=6) as pool:
    results: list[tuple[list[GroundTruthRecord], ResponseUsage]] = (
        map_progress(pool, documents, generate_ground_truth))

  0%|          | 0/85 [00:00<?, ?it/s]

In [18]:
ground_truth: list[GroundTruthRecord] = []
usages: list[ResponseUsage] = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

425

In [19]:
from lib.llm import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost.total_cost

total_cost

0.06261375000000001

In [21]:
import pandas as pd


ground_truth_df = pd.DataFrame(ground_truth)

In [24]:
ground_truth_df.to_csv("data/faq_ground_truth.csv", index=False)